# 🌊 Nœud d'essaim gen3 — GPU Colab en production

Ce notebook transforme **CE Colab en un nœud de l'essaim média gen3** : il bat l'onde de vie
exactement comme un navigateur connecté, reçoit des **parts de tâches** des demandes réelles
des utilisateurs, les exécute sur **le GPU du Colab**, et renvoie les signaux à l'IA centrale
qui les fusionne (image / voix / vidéo).

**Le GPU de Colab et les GPU des utilisateurs travaillent EN MÊME TEMPS** :
- *image* → **mosaïque grand format 2×2** : 4 tuiles SD réparties sur les GPU disponibles
  (Colab + navigateurs), cousues par l'IA centrale ;
- *vidéo* → 6 keyframes réparties sur les 5 meilleurs GPU (Colab + navigateurs) ;
- *voix* → segments de phrases répartis sur l'essaim.

## ⚠️ Loi du propriétaire (CONTRAINTE-ONDES)
> *« Les IA ajoutées à gen3 ne sont que temporaires : le GPU de Colab ne doit servir qu'à
> traiter le signal d'onde de gen3. »*

Ce notebook respecte la loi : les exécuteurs média ci-dessous sont **l'échafaudage temporaire**
de la pipeline d'ondes média de gen3 (les mêmes poids/famille de modèles que les moteurs
WebGPU du navigateur — `a_publier_apres_colab`), et la cellule 5 implémente le
**traitement du signal d'onde gen3 lui-même (RPIA/W-DNN)** en torch. Après distillation,
les poids vivront dans le navigateur et cet échafaudage sera supprimé.

## Honnêteté d'ingénierie (limites Colab)
- Colab est un environnement **éphémère** (session max ~12 h, déconnexion sur inactivité,
  quotas quotidiens) : ce n'est PAS un serveur 24/7. Tant que le notebook tourne, il sert
  l'essaim ; dès qu'il s'arrête, la sentinelle centrale **relance automatiquement** ses
  parts vers d'autres GPU — l'utilisateur n'est jamais livré d'un faux succès.
- Le premier job image/vidéo télécharge les poids SD (~5 Go, 1–3 min). C'est normal.
- Configuration requise côté plateforme : la variable d'environnement `ESSAIM_NOEUD_TOKEN`
  doit être définie sur le serveur (sinon la voie machine est désactivée, fail-closed).

In [ ]:
#@title 1 · Configuration (jeton + identité du nœud)
import getpass, hashlib

GEN3_URL = "https://gen3ia.online"  #@param {type:"string"}

GEN3_NOEUD_TOKEN = ""
try:  # Colab Secrets : icône 🔑 à gauche → « GEN3_NOEUD_TOKEN »
    from google.colab import userdata
    GEN3_NOEUD_TOKEN = (userdata.get("GEN3_NOEUD_TOKEN") or "").strip()
    print("Jeton lu depuis les Secrets Colab.")
except Exception:
    pass
if not GEN3_NOEUD_TOKEN:
    GEN3_NOEUD_TOKEN = getpass.getpass("Collez le jeton du nœud (ESSAIM_NOEUD_TOKEN) : ").strip()

assert len(GEN3_NOEUD_TOKEN) >= 24, "Jeton trop court — demandez le jeton ESSAIM_NOEUD_TOKEN au propriétaire."

# Identité STABLE du nœud : dérivée du jeton (le même jeton = le même nœud,
# la réputation de succès s'accumule entre les sessions).
empreinte = hashlib.sha256(GEN3_NOEUD_TOKEN.encode()).hexdigest()[:12]
NOEUD_ID = f"colab-{empreinte}"
print(f"Nœud : {NOEUD_ID}  →  {GEN3_URL}")

In [ ]:
#@title 2 · Dépendances (torch est déjà sur Colab)
!pip -q install "diffusers>=0.31" "transformers>=4.44" accelerate safetensors "kokoro>=0.9.2" soundfile requests > /dev/null
!apt-get -qq install -y espeak-ng > /dev/null 2>&1 || true
print("Dépendances prêtes.")

In [ ]:
#@title 3 · Sonde GPU — capacités annoncées à l'IA centrale
import torch

assert torch.cuda.is_available(), "⚠️ Activez le GPU : Exécution ▸ Modifier le type d'exécution ▸ T4/L4 GPU"

prop = torch.cuda.get_device_properties(0)
VRAM_GO = round(prop.total_memory / 1e9, 1)
ADAPTATEUR = f"{prop.name} (Colab CUDA)"

# « webgpu: true » est le drapeau PROTOCOLE de l'essaim : « ce nœud peut
# exécuter des fragments ». Ici l'exécution est CUDA/torch — le nom
# d'adaptateur et les features disent la vérité (Colab, pas un navigateur).
CAPACITÉS = {
    "webgpu": True,
    "memoireGo": VRAM_GO,
    "adaptateur": ADAPTATEUR,
    "features": ["colab", "cuda", "float16"],
    "appareil": "colab-notebook",
}
print(f"GPU : {ADAPTATEUR} — {VRAM_GO} Go")

In [ ]:
#@title 4 · Traitement du signal d'onde gen3 (LOI CONTRAINTE-ONDES) — RPIA en torch
import math, torch

def hachage_prompt(prompt: str) -> int:
    """Portage fidèle de `hachagePrompt` (lib/ondes/encodeur) : LCG-JS. """
    h = 0
    for ch in prompt:
        h = ((h << 5) - h + ord(ch)) & 0xFFFFFFFF
    if h >= 0x80000000:
        h -= 1 << 32
    return h

def js_mod(dividende: int, diviseur: int) -> int:
    r = abs(dividende) % diviseur
    return -r if dividende < 0 else r

def encoder_texte_en_onde_cible(prompt: str, nb_noeuds: int) -> torch.Tensor:
    """Le prompt devient une ONDE CIBLE (fréquence de base + harmoniques)."""
    h = hachage_prompt(prompt)
    frequence_base = abs(js_mod(h, 1000)) + 100
    enveloppe = math.cos(h / 10000)
    onde = [
        math.sin((i * frequence_base * ((i % 10) + 1)) / nb_noeuds) * enveloppe
        for i in range(nb_noeuds)
    ]
    return torch.tensor(onde, dtype=torch.float32)


class EntraîneurRPIA:
    """MIRROIR torch (GPU) de `SHADER_ENTRAINEMENT_RPIA` / EntraineurRPIA.ts.

    N nœuds [amplitude, fréquence, phase, amortissement] apprennent à
    résonner vers l'onde cible :
      · |interférence| < 0.05 → constructive : phase += lr·(1−cos φ),
        amortissement relâché (−0.002, plancher 0.001) ;
      · sinon → destructive : phase −= lr·sin(interf),
        amortissement renforcé (+0.005, plafond 0.5) ;
      · amplitude −= lr·interférence·0.1, bornée [0, 2].
    Graine 42, LCG identique à la source JS : même réseau initial.
    """

    def __init__(self, nb_noeuds: int, graine: int = 42, amplitude_initiale: float = 1.0):
        assert nb_noeuds >= 1
        self.nb_noeuds = nb_noeuds
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        g = graine & 0xFFFFFFFF

        def alea() -> float:
            nonlocal g
            g = (g * 1664525 + 1013904223) & 0xFFFFFFFF
            return g / 0xFFFFFFFF

        self.amplitude = torch.full((nb_noeuds,), float(amplitude_initiale), device=self.device)
        self.frequence = torch.tensor([0.03 + alea() * 0.14 for _ in range(nb_noeuds)], device=self.device)
        self.phase = torch.tensor([alea() * math.pi * 2 for _ in range(nb_noeuds)], device=self.device)
        self.amortissement = torch.full((nb_noeuds,), 0.05, device=self.device)
        self.compteur_pas = 0

    def pas_entrainement(self, cible: torch.Tensor, lr: float = 0.01, pas_de_temps: float = 0.1) -> float:
        temps = pas_de_temps * self.compteur_pas
        onde = self.amplitude * torch.cos(self.frequence * temps + self.phase)
        interf = onde - cible.to(self.device)
        constructif = interf.abs() < 0.05
        self.phase = torch.where(
            constructif,
            self.phase + lr * (1.0 - torch.cos(self.phase)),
            self.phase - lr * torch.sin(interf),
        )
        self.amortissement = torch.where(
            constructif,
            (self.amortissement - 0.002).clamp(min=0.001),
            (self.amortissement + 0.005).clamp(max=0.5),
        )
        self.amplitude = (self.amplitude - lr * interf * 0.1).clamp(0.0, 2.0)
        self.compteur_pas += 1
        onde_après = self.amplitude * torch.cos(self.frequence * temps + self.phase)
        return (onde_après - cible.to(self.device)).abs().mean().item()

    def entrainer(self, cible: torch.Tensor, epoques: int, lr: float = 0.01, pas_de_temps: float = 0.1):
        historique = [self.pas_entrainement(cible, lr, pas_de_temps) for _ in range(epoques)]
        return historique

    def exporter_poids(self):
        """[amplitude, fréquence, phase, amortissement] × N — format packaging."""
        return torch.stack([self.amplitude, self.frequence, self.phase, self.amortissement], dim=1) \
            .reshape(-1).cpu().tolist()


# Preuve sur GPU : 100 nœuds apprennent l'onde cible du prompt de test.
ent = EntraîneurRPIA(100)
cible = encoder_texte_en_onde_cible("gen3 traite son signal d'onde", 100)
courbe = ent.entrainer(cible, 300)
print(f"RPIA GPU : erreur {courbe[0]:.4f} → {courbe[-1]:.4f} sur 300 pas "
      f"({ent.device}) — poids exportables : {len(ent.exporter_poids())} valeurs.")

In [ ]:
#@title 5 · Exécuteurs média (échafaudage temporaire de la pipeline d'ondes gen3)
import io

DIMENSIONS = {  # miroir exact de dimensionsPourRatio (lib/essaim-media)
    "1:1": (512, 512), "3:4": (448, 576), "4:3": (576, 448), "16:9": (576, 320),
    "9:16": (320, 576), "2:3": (416, 608), "3:2": (608, 416), "21:9": (640, 288),
}
MODEL_SD = "stabilityai/stable-diffusion-2-1-base"  # famille des poids a_publier_apres_colab
_cache = {}


def pipeline_image():
    if "sd" not in _cache:
        from diffusers import StableDiffusionPipeline
        print("⏳ Téléchargement des poids SD (une seule fois, ~5 Go)…")
        pipe = StableDiffusionPipeline.from_pretrained(
            MODEL_SD, torch_dtype=torch.float16,
            safety_checker=None, requires_safety_checker=False,
        ).to("cuda")
        pipe.set_progress_bar_config(disable=True)
        _cache["sd"] = pipe
    return _cache["sd"]


def exécuter_image(charge: dict, largeur=None, hauteur=None) -> bytes:
    w = int(largeur or DIMENSIONS.get(charge.get("ratio", "1:1"), (512, 512))[0])
    h = int(hauteur or DIMENSIONS.get(charge.get("ratio", "1:1"), (512, 512))[1])
    pipe = pipeline_image()
    g = torch.Generator(device="cuda").manual_seed(int(charge["seed"]) % (2 ** 31))
    img = pipe(
        charge["prompt"], width=w, height=h,
        num_inference_steps=int(charge.get("steps", 26)),
        guidance_scale=float(charge.get("guidance", 5.0)),
        generator=g,
    ).images[0]
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return buf.getvalue()


def pipeline_voix():
    if "kokoro" not in _cache:
        from kokoro import KPipeline
        print("⏳ Téléchargement des poids Kokoro (une seule fois)…")
        _cache["kokoro"] = KPipeline(lang_code="f")  # français
    return _cache["kokoro"]


def exécuter_segment(charge: dict) -> bytes:
    import soundfile as sf
    pipe = pipeline_voix()
    morceaux = [
        torch.from_numpy(audio)
        for _, _, audio in pipe(
            charge["texte"],
            voice=charge.get("voix", "ff_siwis"),
            speed=float(charge.get("vitesse", 1.0)),
        )
    ]
    audio = torch.cat(morceaux).cpu().numpy() if morceaux else None
    if audio is None or len(audio) == 0:
        raise RuntimeError("Synthèse vocale vide")
    buf = io.BytesIO()
    sf.write(buf, audio, 24000, format="WAV", subtype="PCM_16")
    return buf.getvalue()


print("Exécuteurs prêts (chargement paresseux des poids au premier job).")

In [ ]:
#@title 6 · Protocole essaim — battement d'onde, exécution, retour des signaux
import time, requests

session = requests.Session()
session.headers["authorization"] = f"Bearer {GEN3_NOEUD_TOKEN}"


def battement(fragment_en_cours: str | None = None) -> dict:
    corps = {"noeudId": NOEUD_ID, "capacités": CAPACITÉS}
    if fragment_en_cours:
        corps["fragmentId"] = fragment_en_cours  # « je commence réellement cette part »
    r = session.post(f"{GEN3_URL}/api/essaim/ondes", json=corps, timeout=20)
    r.raise_for_status()
    return r.json()


def livrer(fragment_id: str, octets: bytes, nom: str, mime: str) -> dict:
    r = session.post(
        f"{GEN3_URL}/api/essaim/resultat",
        data={"noeudId": NOEUD_ID, "fragmentId": fragment_id},
        files={"file": (nom, octets, mime)},
        timeout=90,
    )
    return r.json()


def échouer(fragment_id: str, raison: str) -> dict:
    r = session.post(
        f"{GEN3_URL}/api/essaim/resultat",
        data={"noeudId": NOEUD_ID, "fragmentId": fragment_id, "echec": "1", "raison": raison[:300]},
        timeout=30,
    )
    return r.json()


def exécuter_affectation(frag: dict) -> dict:
    sous = frag["charge"].get("sous")
    print(f"⚡ Part {frag['index'] + 1}/{frag['total']} [{frag['kind']}·{sous}] "
          f"job {frag['jobId'][:8]}…")
    battement(frag["fragmentId"])  # annonce d'exécution réelle
    if sous in ("image", "keyframe", "tuile"):
        octets = exécuter_image(frag["charge"], frag["charge"].get("largeur"), frag["charge"].get("hauteur"))
        nom = {"image": "image.png", "keyframe": "keyframe.png", "tuile": "tuile.png"}[sous]
        return livrer(frag["fragmentId"], octets, nom, "image/png")
    if sous == "segment":
        return livrer(frag["fragmentId"], exécuter_segment(frag["charge"]), "segment.wav", "audio/wav")
    raise ValueError(f"Sous-charge inconnue : {sous}")


def boucle_de_vie(période_s: float = 12.0) -> None:
    """Onde de vie : bat toutes les ~12 s, exécute les parts confiées. Ctrl-C pour partir."""
    print(f"🌊 {NOEUD_ID} rejoint l'essaim ({ADAPTATEUR}, {VRAM_GO} Go) — Ctrl-C pour un départ propre.")
    while True:
        try:
            rep = battement()
            for frag in rep.get("affectations", []):
                try:
                    print("  ✓", exécuter_affectation(frag))
                except KeyboardInterrupt:
                    raise
                except Exception as e:  # échec HONNÊTE : la centrale relance la part ailleurs
                    print("  ✗", e)
                    try:
                        échouer(frag["fragmentId"], str(e))
                    except Exception:
                        pass
            noeud = rep.get("noeud", {})
            print(f"🌊 score {noeud.get('score')} · charge {noeud.get('charge')} · "
                  f"essaim {rep.get('essaim', {}).get('noeudsEnLigne')} nœud(s)")
        except KeyboardInterrupt:
            print("Départ propre : onde d'adieu envoyée à l'IA centrale.")
            try:
                session.delete(f"{GEN3_URL}/api/essaim/ondes", json={"noeudId": NOEUD_ID}, timeout=10)
            except Exception:
                pass
            break
        except Exception as e:
            print("⚠ battement raté (réseau/serveur) — nouvelle onde dans la prochaine période :", e)
        time.sleep(période_s)

In [ ]:
#@title 7 · 🚀 LANCER LE NŒUD (interrompre avec ⏹ / Ctrl-C pour un départ propre)
boucle_de_vie()

## Arrêt et comportement après coupure
- **Ctrl-C / ⏹** → l'onde d'adieu (`DELETE /api/essaim/ondes`) prévient l'IA centrale : vos parts
  ne sont jamais perdues.
- **Déconnexion brutale Colab** (quota, timeout) → la sentinelle centrale détecte la rupture
  (silence > 26 s) et **relance automatiquement** les parts vers les autres GPU de l'essaim
  (navigateurs ou un futur nœud Colab). L'utilisateur reçoit toujours son œuvre.

## Ce que ce nœud change pour les utilisateurs réels
| Demande depuis l'UI | GPU Colab | GPU de l'utilisateur | Fusion (IA centrale) |
|---|---|---|---|
| Image | ~2 tuiles / 4 (mosaïque 2×2) | ~2 tuiles / 4 (WebGPU) | couture des tuiles |
| Vidéo | ~3 keyframes / 6 | ~3 keyframes / 6 (WebGPU) | MP4 fondus |
| Voix | segments alternés | segments alternés | WAV continu |

## Après la distillation (loi du propriétaire)
Les poids d'onde RPIA entraînés ici (cellule 4) sont exportables
(`exporter_poids()` → `[amplitude, fréquence, phase, amortissement] × N`) pour le packaging
`a_publier_apres_colab` : le navigateur reprendra le traitement du signal, et cet échafaudage
sera retiré sans toucher gen3.